# 1 - Creating event dataset for scenario reconstruction from CICIDS2017

## 1.1 - Creating machine learning datasets

### 1.1.1 - Validate files

In [ ]:
# Make project helpers available from a kernel started in this folder or the repo root.
from pathlib import Path
import sys
PROJECT_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents)
                    if (root / "preprocessing").is_dir() and (root / "scenario_reconstructor").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
print(os.getcwd())
for root, dirs, files in os.walk('./data'):
        print(root+":")
        if dirs:
            print('\n'.join(dirs))
        if files:
            print('\n'.join(files))

### 1.1.2 - Dataset loading

In [ ]:
import pandas as pd
import numpy as np
df_train=pd.read_csv("./data/train.csv")
df_test=pd.read_csv("./data/test.csv")

In [ ]:
df_test["index"] = np.arange(len(df_test))

### 1.1.3 - Dataset cleaning

In [ ]:
df_train.columns = df_train.columns.str.strip()
df_test.columns = df_test.columns.str.strip()
initial_columns = df_train.columns
for column in initial_columns:
    if any(keyword in column for keyword in ["id", "ip", "mac", "oui", "duration", "seen", "protocol", "Unnamed", "fingerprint", "timestamp", "phase", "step"]):
        df_train = df_train.drop(columns=column)
        df_test = df_test.drop(columns=column, errors="ignore")

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
import numpy as np
df_train = df_train.dropna(thresh=2, axis=1).dropna(thresh=2, axis=0)
# Do not derive a different feature set from evaluation data.
df_test = df_test.dropna(thresh=2, axis=0)

In [ ]:
num_cols = df_train.select_dtypes(include="number").columns
df_train[num_cols] = df_train[num_cols].replace([np.inf, -np.inf, np.nan], -1)

str_cols = df_train.select_dtypes(exclude="number").columns
df_train[str_cols] = df_train[str_cols].fillna(" ")

In [ ]:
num_cols = df_test.select_dtypes(include="number").columns
df_test[num_cols] = df_test[num_cols].replace([np.inf, -np.inf, np.nan], -1)

str_cols = df_test.select_dtypes(exclude="number").columns
df_test[str_cols] = df_test[str_cols].fillna(" ")

In [ ]:
print(df_train.isna().sum().sum())

In [ ]:
print(df_test.isna().sum().sum())

In [ ]:
THRESHOLD = 0.99

In [ ]:
low_var_cols = [col for col in df_train.columns
                if df_train[col].value_counts(normalize=True, dropna=False).iloc[0] >= THRESHOLD]
df_train = df_train.drop(columns=low_var_cols)
df_test = df_test.drop(columns=low_var_cols, errors="ignore")

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
from preprocessing.training import align_test_schema
df_test = align_test_schema(df_train, df_test)

In [ ]:
df_train["label"].value_counts()

In [ ]:
df_test["label"].value_counts()

In [ ]:
from preprocessing.training import filter_known_test_classes
df_test = filter_known_test_classes(df_train, df_test)

In [ ]:
df_train["label"].value_counts()

In [ ]:
df_test["label"].value_counts()

### 1.1.7 - Saving train-set and test-set

In [ ]:
df_train.to_csv("./data/ML_train.csv", index=False)
df_test.to_csv("./data/ML_test.csv", index=False)

## 1.2 - Computing anomaly score

### 1.2.1 - Loading the datasets

In [ ]:
import numpy as np
import pandas as pd
df_train=pd.read_csv("./data/ML_train.csv")
df_test=pd.read_csv("./data/ML_test.csv")
ind_test=df_test["index"]

### 1.2.2 - Categorical encoding for anomaly detection

In [ ]:
attack_labels=["nmap_10_T5", "nmap_banner", "nmap_mqtt", "brute_force_malformed", "dollar_char", "scp_inst"]

In [ ]:
from preprocessing.training import encode_binary_labels
label_encode = encode_binary_labels

In [ ]:
from preprocessing.training import apply_label_encoding
label_encode_categories = apply_label_encoding

In [ ]:
print(df_train.isna().sum().sum())

In [ ]:
print(df_test.isna().sum().sum())

In [ ]:
from preprocessing.training import split_training_validation

# Reproducible calibration holdout from training data; final test data is untouched.
df_fit, df_validation = split_training_validation(df_train, attack_labels)
df_train_encoded, categories = label_encode(df_fit, attack_labels)
df_validation_encoded = label_encode_categories(df_validation, attack_labels, categories)
df_test_encoded = label_encode_categories(df_test, attack_labels, categories)

In [ ]:
df_train_encoded.info()

In [ ]:
df_test_encoded.info()

In [ ]:
print(df_train_encoded.isna().sum().sum())

In [ ]:
print(df_test_encoded.isna().sum().sum())

### 1.2.3 - Feature label separation

In [ ]:
y_train, X_train = df_train_encoded["label"], df_train_encoded.drop("label", axis=1)
y_validation, X_validation = df_validation_encoded["label"], df_validation_encoded.drop("label", axis=1)
y_test, X_test = df_test_encoded["label"], df_test_encoded.drop(["label", "index"], axis=1)
X_train.info()
X_validation.info()
X_test.info()

### 1.2.4 - Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_validation = scaler.transform(X_validation)
X_test = scaler.transform(X_test)

### 1.2.5 - Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestRegressor
regressor=RandomForestRegressor(random_state=42, n_jobs=-1)
regressor.fit(X_train, y_train)

In [ ]:
for i in range(X_train.shape[1]):
    print(f"{df_train_encoded.columns[i]}: {np.round(regressor.feature_importances_[i],3)}")

### 1.2.7 - Computing contamination

In [ ]:
print("contamination: ")
print(y_train.value_counts().get(1)/len(y_train))

### 1.2.8 - Train anomaly detectors

In [ ]:
from pyod.models.iforest import IForest
iforest=IForest(contamination=0.46, n_estimators=500,  n_jobs=10, random_state=42)
iforest.fit(X_train)

In [ ]:
from pyod.models.hbos import HBOS
hbos=HBOS(n_bins=500, contamination=0.46)
hbos.fit(X_train)

In [ ]:
from pyod.models.sampling import Sampling
sampling=Sampling(contamination=0.46, subset_size=200, random_state=42)
sampling.fit(X_train)

In [ ]:
from sklearn.naive_bayes import GaussianNB
naive_bayes=GaussianNB()
naive_bayes.fit(X_train, y_train)

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
qda = QuadraticDiscriminantAnalysis(reg_param=1.0)
qda.fit(X_train, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression
logistic_regression=LogisticRegression(max_iter=500, random_state=42)
logistic_regression.fit(X_train, y_train)

In [ ]:
from xgboost import XGBClassifier
xgb=XGBClassifier(n_estimators=50, random_state=42)
xgb.fit(X_train,y_train)

### 1.2.9 - Test anomaly detection function definition

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
def evaluate_metrics(predictor, name):
    y_pred=predictor.predict(X_test)
    acc=accuracy_score(y_test, y_pred)
    print(f"{name} test accuracy: {acc:.3f}")
    rec=recall_score(y_test, y_pred)
    print(f"{name} test recall: {rec:.3f}")
    prec=precision_score(y_test, y_pred)
    print(f"{name} test precision: {prec:.3f}")

### 1.2.10 - ROC and Precision-Recall Functions definition

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt

def show_roc(predictor, name):
    y_prob = predictor.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))

    RocCurveDisplay(
        fpr=fpr,
        tpr=tpr,
        roc_auc=roc_auc,
        estimator_name=name
    ).plot(ax=ax)

    ax.set_title(name)
    ax.grid(True)

    fig.suptitle(f"ROC Curve ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(f"roc_{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

    return y_prob

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    PrecisionRecallDisplay,
)
import matplotlib.pyplot as plt

def show_prc(predictor, name):
    y_prob = predictor.predict_proba(X_test)[:, 1]

    pre, rec, _ = precision_recall_curve(y_test, y_prob)
    avg_pre = average_precision_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))

    PrecisionRecallDisplay(
        precision=pre,
        recall=rec,
        average_precision=avg_pre,
        estimator_name=name
    ).plot(ax=ax)

    ax.set_title(name)
    ax.grid(True)

    fig.suptitle(f"Precision-Recall Curve ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(f"prc_{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

    return y_prob

In [ ]:
from preprocessing.training import calibrate_threshold

def compute_thresh(predictor, recall):
    threshold = calibrate_threshold(predictor, X_validation, y_validation, recall)
    print(f"Computed threshold: {threshold}")
    return threshold

### 1.2.11 - Test anomaly detectors

In [ ]:
evaluate_metrics(iforest, "IForest")

In [ ]:
show_roc(iforest, "IForest")

In [ ]:
show_prc(iforest, "IForest")

In [ ]:
evaluate_metrics(hbos, "HBOS")

In [ ]:
show_roc(hbos, "HBOS")

In [ ]:
show_prc(hbos, "HBOS")

In [ ]:
evaluate_metrics(sampling, "Sampling")

In [ ]:
show_roc(sampling, "Sampling")

In [ ]:
show_prc(sampling, "Sampling")

In [ ]:
evaluate_metrics(naive_bayes, "GaussianNB")

In [ ]:
show_roc(naive_bayes, "GaussianNB")

In [ ]:
show_prc(naive_bayes, "GaussianNB")

In [ ]:
evaluate_metrics(qda, "QuadraticDiscriminantAnalysis")

In [ ]:
show_roc(qda, "QuadraticDiscriminantAnalysis")

In [ ]:
show_prc(qda, "QuadraticDiscriminantAnalysis")

In [ ]:
evaluate_metrics(logistic_regression, "LogisticRegression")

In [ ]:
show_roc(logistic_regression, "LogisticRegression")

In [ ]:
show_prc(logistic_regression, "LogisticRegression")

In [ ]:
evaluate_metrics(xgb, "XGBClassifier")

In [ ]:
show_roc(xgb, "XGBClassifier")

In [ ]:
show_prc(xgb, "XGBClassifier")

### 1.2.12 - Computing target threshold

In [ ]:
print("Isolation Forest")
compute_thresh(iforest, 0.8)

In [ ]:
print("Logistic Regression")
compute_thresh(logistic_regression, 0.8)

In [ ]:
print("XGBoost")
compute_thresh(xgb, 0.8)

In [ ]:
print("Quadratic Discriminant Analysis")
compute_thresh(qda, 0.8)

In [ ]:
print("Naive Bayes")
compute_thresh(naive_bayes, 0.8)

In [ ]:
print("Sampling")
compute_thresh(sampling, 0.8)

In [ ]:
print("HBOS")
compute_thresh(hbos, 0.8)

### 1.2.13 - Computing anomaly score 

In [ ]:
anomaly_score_dict={}
anomaly_score_dict["logistic_regression"]=logistic_regression.predict_proba(X_test)[:,1]
anomaly_score_dict["xgb"]=xgb.predict_proba(X_test)[:,1]
anomaly_score_dict["qda"]=qda.predict_proba(X_test)[:,1]
anomaly_score_dict["naive_bayes"]=naive_bayes.predict_proba(X_test)[:,1]
anomaly_score_dict["sampling"]=sampling.predict_proba(X_test)[:,1]
anomaly_score_dict["hbos"]=hbos.predict_proba(X_test)[:,1]
anomaly_score_dict["iforest"]=iforest.predict_proba(X_test)[:,1]
anomaly_score_dict["perfect"]=y_test

In [ ]:
for key, df in anomaly_score_dict.items():
    print(key+ f": {df.size}")

## 1.3 - Computing attack type score

### 1.3.1 - Loading the datasets

In [ ]:
import numpy as np
import pandas as pd
df_train=pd.read_csv("./data/ML_train.csv")
df_test=pd.read_csv("./data/ML_test.csv")
ind_test=df_test["index"]

### 1.3.2 - Categorical encoding for type mapping

In [ ]:
df_dict_train={}
df_dict_test={}
attack_types=[label for label, value in df_train['label'].value_counts().items() if label!='normal']
print(attack_types)
for label in attack_types:
    df_dict_train[label], categories=label_encode(df_train, [label])
    df_dict_test[label]=label_encode_categories(df_test, [label], categories)

In [ ]:
for label in attack_types:
    print(label)
    print("train set")
    print(df_dict_train[label]["label"].value_counts())
    print("test set")
    print(df_dict_test[label]["label"].value_counts())

### 1.3.4 - Feature label separation

In [ ]:
y_train_dict={}
X_train_dict={}
y_test_dict={}
X_test_dict={}

for label in attack_types:
    y_train_dict[label], X_train_dict[label]= df_dict_train[label]["label"], df_dict_train[label].drop("label", axis=1)
    y_test_dict[label], X_test_dict[label]= df_dict_test[label]["label"], df_dict_test[label].drop(["label", "index"], axis=1)
    X_train_dict[label].info()
    X_test_dict[label].info()

### 1.3.5 - Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
for label in attack_types:
    scaler = StandardScaler()
    scaler.fit(X_train_dict[label])
    X_train_dict[label] = scaler.transform(X_train_dict[label])
    X_test_dict[label] = scaler.transform(X_test_dict[label])

### 1.3.5 - Train type mappers

In [ ]:
from xgboost import XGBClassifier
xgb_dict={}
for label in attack_types:
    xgboost=XGBClassifier(n_estimators=50, random_state=42)
    xgboost.fit(X_train_dict[label], y_train_dict[label])
    xgb_dict[label]=xgboost

In [ ]:
from sklearn.naive_bayes import GaussianNB
naive_bayes_dict={}
for label in attack_types:
    nb=GaussianNB()
    nb.fit(X_train_dict[label], y_train_dict[label])
    naive_bayes_dict[label]=nb

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
qda_dict={}
for label in attack_types:
    qda = QuadraticDiscriminantAnalysis(solver='eigen', shrinkage=1.0)
    qda.fit(X_train_dict[label], y_train_dict[label])
    qda_dict[label]=qda

In [ ]:
from sklearn.linear_model import LogisticRegression
logistic_regression_dict={}
for label in attack_types:
    logistic_regression=LogisticRegression(max_iter=500, random_state=42)
    logistic_regression.fit(X_train_dict[label], y_train_dict[label])
    logistic_regression_dict[label]=logistic_regression

### 1.3.6 - Test type mapper function definition

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
def evaluate_mapper(mapper_dict, name):
    for label, xgboost in mapper_dict.items(): 
        y_pred=xgboost.predict(X_test_dict[label])
        acc=accuracy_score(y_test_dict[label], y_pred)
        print(f"{name} test accuracy for {label}: {acc:.3f}")
        rec=recall_score(y_test_dict[label], y_pred)
        print(f"{name} test recall for {label}: {rec:.3f}")
        rec=precision_score(y_test_dict[label], y_pred)
        print(f"{name} test precision for {label}: {rec:.3f}")

### 1.3.7 - Mapper ROC curve function definition

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt
import math

def show_roc_mapper(mapper_dict, name):
    prob_dict = {}

    n_plots = len(mapper_dict)
    n_cols = 3
    n_rows = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows)
    )

    # Flatten axes for easy indexing
    if n_plots == 1:
        axes = [axes]
    else:
        axes = axes.ravel()

    for ax, (label, predictor) in zip(axes, mapper_dict.items()):
        y_prob = predictor.predict_proba(X_test_dict[label])[:, 1]
        prob_dict[label] = y_prob

        fpr, tpr, _ = roc_curve(y_test_dict[label], y_prob)
        roc_auc = roc_auc_score(y_test_dict[label], y_prob)

        RocCurveDisplay(
            fpr=fpr,
            tpr=tpr,
            roc_auc=roc_auc,
            estimator_name=f"{name} {label}"
        ).plot(ax=ax)

        ax.set_title(f"{label}")
        ax.grid(True)

    # Hide any unused subplots
    for ax in axes[n_plots:]:
        ax.set_visible(False)

    fig.suptitle(f"ROC Curves ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(f"roc_{name}_mapper.png", dpi=300, bbox_inches="tight")

    plt.show()

    return prob_dict

### 1.3.8 - Mapper Precision-Recall curve function definition

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    PrecisionRecallDisplay,
)
import matplotlib.pyplot as plt
import math

def show_prc_mapper(mapper_dict, name):
    prob_dict = {}

    n_plots = len(mapper_dict)
    n_cols = 3
    n_rows = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows)
    )

    if n_plots == 1:
        axes = [axes]
    else:
        axes = axes.ravel()

    for ax, (label, predictor) in zip(axes, mapper_dict.items()):
        y_prob = predictor.predict_proba(X_test_dict[label])[:, 1]
        prob_dict[label] = y_prob

        pre, rec, _ = precision_recall_curve(
            y_test_dict[label],
            y_prob
        )
        avg_pre = average_precision_score(
            y_test_dict[label],
            y_prob
        )

        PrecisionRecallDisplay(
            precision=pre,
            recall=rec,
            average_precision=avg_pre,
            estimator_name=f"{name} {label}"
        ).plot(ax=ax)

        ax.set_title(f"{label}")
        ax.grid(True)

    for ax in axes[n_plots:]:
        ax.set_visible(False)

    fig.suptitle(f"Precision-Recall Curves ({name})", fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(f"prc_{name}_mapper.png", dpi=300, bbox_inches="tight")

    plt.show()

    return prob_dict

### 1.3.9 - Test attack type mappers

In [ ]:
evaluate_mapper(xgb_dict,"XGBoostClassifier")

In [ ]:
show_roc_mapper(xgb_dict,"XGBoostClassifier")

In [ ]:
show_prc_mapper(xgb_dict,"XGBoostClassifier")

In [ ]:
evaluate_mapper(naive_bayes_dict,"GaussianNB")

In [ ]:
show_roc_mapper(naive_bayes_dict,"GaussianNB")

In [ ]:
show_prc_mapper(naive_bayes_dict,"GaussianNB")

In [ ]:
evaluate_mapper(qda_dict,"QuadraticDiscriminantAnalysis")

In [ ]:
show_roc_mapper(qda_dict,"QuadraticDiscriminantAnalysis")

In [ ]:
show_prc_mapper(qda_dict,"QuadraticDiscriminantAnalysis")

In [ ]:
evaluate_mapper(logistic_regression_dict,"LogisticRegression")

In [ ]:
show_roc_mapper(logistic_regression_dict,"LogisticRegression")

In [ ]:
show_prc_mapper(logistic_regression_dict,"LogisticRegression")

### 1.3.10 - Computing attack type score

In [ ]:
xgb_attack_type_prob={}

In [ ]:
# choose the attack type mapper you want to use
for label, xgboost in xgb_dict.items():
    y_prob = xgboost.predict_proba(X_test_dict[label])[:, 1]
    xgb_attack_type_prob[label]=y_prob

In [ ]:
for key, prob_vector in xgb_attack_type_prob.items():
    print(key+f":{prob_vector.shape[0]}")

In [ ]:
gnb_attack_type_prob={}

In [ ]:
# choose the attack type mapper you want to use
for label, gaussiannb in naive_bayes_dict.items():
    y_prob = gaussiannb.predict_proba(X_test_dict[label])[:, 1]
    gnb_attack_type_prob[label]=y_prob

In [ ]:
for key, prob_vector in gnb_attack_type_prob.items():
    print(key+f":{prob_vector.shape[0]}")

In [ ]:
perfect_type_prob={}

In [ ]:
# choose the attack type mapper you want to use
for label, score in y_test_dict.items():
    perfect_type_prob[label] = score

In [ ]:
for key, prob_vector in perfect_type_prob.items():
    print(key+f":{prob_vector.shape[0]}")

## 1.4 - Building the scenario reconstruction dataset

### 1.4.1 - Loading the starting dataset 

In [ ]:
import numpy as np
import pandas as pd
df_rec=pd.read_csv("data/test.csv")
df_rec["index"] = pd.Series(np.arange(len(df_rec)))
df_rec.info()

In [ ]:
import numpy as np
df_rec = df_rec.dropna(thresh=2, axis=0)
df_rec = df_rec.dropna(thresh=2, axis=0)

In [ ]:
df_rec=df_rec[df_rec["label"].isin(df_train["label"].unique())]
df_rec=df_rec[df_rec["label"].isin(df_test["label"].unique())]

In [ ]:
attack_steps = (
    df_rec
    .groupby('label', as_index=False)
    .agg(total_steps=('step_number', 'nunique'))
)

print(attack_steps)

In [ ]:
print(df_rec.shape[0])

### 1.4.2 - Further step subdivision

In [ ]:
attack = df_rec["step_number"] != -1
groups = (
    df_rec.loc[attack]
      .groupby(["step_number", "src_ip", "dst_ip"], sort=False)
      .ngroup()
)
df_rec.loc[attack, "step_number"] = groups

In [ ]:
attack_steps = (
    df_rec[df_rec["label"]!="normal"]
    .groupby('label', as_index=False)
    .agg(total_steps=('step_number', 'nunique'))
)

In [ ]:
attack_steps

In [ ]:
df = df_rec.sort_values('timestamp').reset_index(drop=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df=df[df["label"]!="normal"]

step_summary = (
    df.groupby('step_number', as_index=False)
      .agg(
          label        = ('label', 'first'),
          num_flws     = ('timestamp', 'size'),
          start_time   = ('timestamp', 'min'),
          end_time     = ('timestamp', 'max')
      )
      .assign(duration=lambda g: g['end_time'] - g['start_time'])
)

step_summary['fps'] = (
    step_summary['num_flws'] /
    step_summary['duration'].dt.total_seconds().replace(0, np.nan)
)

label_summary = (
    step_summary.groupby('label', as_index=False)
               .agg(
                   avg_num_flws    = ('num_flws', 'mean'),
                   median_num_flws = ('num_flws', 'median'),
                   avg_duration    = ('duration', 'mean'),
                   median_duration = ('duration', 'median'),
                   total_steps = ('step_number', 'nunique'),
                   total_flws      = ('num_flws', 'sum'),
                   avg_fps         = ('fps', 'mean')
               )
)

In [ ]:
step_summary

In [ ]:
label_summary

### 1.4.3 - Extracting reconstruction features

In [ ]:
rec_cols=["src_ip", "src_port", "dst_ip", "dst_port", "protocol", "timestamp", "bidirectional_duration_ms", "label", "step_number", "index"]
df_rec=df_rec[rec_cols]
df_rec["duration"]=df_rec["bidirectional_duration_ms"]
df_rec=df_rec.drop("bidirectional_duration_ms", axis=1)

In [ ]:
print(df_rec.shape[0])

### 1.4.4 - Adding new features

In [ ]:
df_rec_dict={}
for name, col in anomaly_score_dict.items(): 
    xgb_df_rec = df_rec.copy()
    anomaly_df = pd.DataFrame({"index": ind_test, "anomaly_score": col})
    xgb_df_rec = xgb_df_rec.merge(anomaly_df, on="index", how="inner", validate="one_to_one")
    for attack_type, type_prob in xgb_attack_type_prob.items():
        type_df=pd.DataFrame({"index": ind_test, attack_type: type_prob})
        xgb_df_rec = xgb_df_rec.merge(type_df, on="index", how="inner", validate="one_to_one")
    xgb_df_rec.info()
    df_rec_dict[name+"_xgb"]=xgb_df_rec

    gnb_df_rec = df_rec.copy()
    anomaly_df = pd.DataFrame({"index": ind_test, "anomaly_score": col})
    gnb_df_rec = gnb_df_rec.merge(anomaly_df, on="index", how="inner", validate="one_to_one")
    for attack_type, type_prob in gnb_attack_type_prob.items():
        type_df=pd.DataFrame({"index": ind_test, attack_type: type_prob})
        gnb_df_rec = gnb_df_rec.merge(type_df, on="index", how="inner", validate="one_to_one")
    gnb_df_rec.info()
    df_rec_dict[name+"_gnb"]=gnb_df_rec

    perfect_df_rec=df_rec.copy()
    anomaly_df = pd.DataFrame({"index": ind_test, "anomaly_score": col})
    perfect_df_rec = perfect_df_rec.merge(anomaly_df, on="index", how="inner", validate="one_to_one")
    for attack_type, type_prob in perfect_type_prob.items():
        type_df=pd.DataFrame({"index": ind_test, attack_type: type_prob})
        perfect_df_rec = perfect_df_rec.merge(type_df, on="index", how="inner", validate="one_to_one")
    perfect_df_rec.info()
    df_rec_dict[name+"_perfect"]=perfect_df_rec

### 1.4.5 - Saving the dataset 

In [ ]:
import pandas as pd
for name, df in df_rec_dict.items():
    df=df.sort_values(by="timestamp")
    df.to_csv(f"data/final datasets/{name}_rec_dataset.csv",index=False)